# EconEnv on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/merwanroudane/econenv/blob/main/examples/11_colab_quickstart.ipynb)

**Python and R in one Colab notebook**, sharing a dataset with no CSV in
between.

---

**Developed by Dr Merwan Roudane**  
GitHub: <https://github.com/merwanroudane>  
Package: <https://pypi.org/project/econenv/1.2.0/> (v1.2.0)  
Repository: <https://github.com/merwanroudane/econenv>

---

## What works on Colab, and what cannot

Colab runs on Linux, and that decides this before anything is installed:

| Engine | On Colab | Why |
|---|---|---|
| **Python** | works | it is the kernel |
| **R** | works | R is already on the Colab image |
| **Stata** | **possible** | Stata for Linux exists and pystata supports it - install it from Google Drive if you hold a Linux licence |
| **EViews** | no | no Linux build, Wine cannot licence it, and EViews forbids remote access |
| **MATLAB** | **possible** | MATLAB for Linux exists and the Engine API supports it - the same licence question as Stata |

**Stata is possible here.** See the installation notes in the repository for the
Drive-based recipe; EconEnv finds a Linux Stata with no configuration at all.

**EViews is not, and cannot be.** There is no Linux build. Under Wine it cannot
read a valid machine ID, so licence activation fails. And driving a Windows copy
from here is ruled out by EViews itself, whose documentation states that *web
server access to EViews via COM is not allowed*. That workaround is easy to
build and contractually prohibited, so EconEnv does not ship it.

For EViews, run the five-engine notebook on a local Windows machine.

---

## Want all five engines, and still Colab?

You can. Colab already runs in a browser **on your own PC**, so point that
interface at a Jupyter server on the same PC: the notebook UI stays Colab,
while the kernel - and therefore Python, R, Stata, EViews **and MATLAB** - is
your Windows machine.

Nothing is exposed to the internet. Your browser talks to `localhost`,
Google's servers never reach your machine, and EViews is driven by local COM
exactly as in a local notebook.

It needs the classic Jupyter stack, because the bridge package dates from
2020 and does not load on notebook 7:

```bash
python -m venv colab-runtime
colab-runtime/Scripts/pip install notebook==6.4.12 jupyter_http_over_ws econenv
colab-runtime/Scripts/jupyter serverextension enable --py jupyter_http_over_ws
colab-runtime/Scripts/jupyter notebook --no-browser --port=8888
```

Then in Colab: the **Connect** arrow, **Connect to a local runtime**, and paste
the `http://localhost:8888/?token=...` URL it printed.

Full instructions and the evidence for that version pin are in
[the installation notes](https://github.com/merwanroudane/econenv/blob/main/docs/installation.md#google-colab).

`%econ doctor` says all of this for you, on the machine you are actually on.

## 1. Install

One line. `%pip` installs into the kernel that is running, which is what
you want inside a notebook.

In [ ]:
%pip install -q econenv

## 2. Load and check

The status table shows what EconEnv found. On Colab you should see Python
and R configured, and EViews reported as unavailable — which is correct,
not a problem to solve.

In [ ]:
%load_ext econenv

In [ ]:
%econ status

`%econ doctor` explains anything that is missing, and on Colab it says
explicitly why Stata and EViews cannot be there.

In [ ]:
%econ doctor

## 3. Python — build a dataset

Real US quarterly macroeconomic data, 1959Q1–2009Q3, shipped with
statsmodels — so nothing is downloaded and nothing is private.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import econenv

raw = sm.datasets.macrodata.load_pandas().data
idx = pd.PeriodIndex(
    year=raw.year.astype(int), quarter=raw.quarter.astype(int), freq='Q'
).to_timestamp()

macro = pd.DataFrame(
    {
        'lrgdp': np.log(raw.realgdp.values),
        'lrcons': np.log(raw.realcons.values),
        'realint': raw.realint.values,
    },
    index=idx,
)
macro.index.name = 'date'
print(len(macro), 'quarters')
macro.head()

## 4. R — the same data, no file in between

`-i macro` sends the DataFrame into R. It arrives as a real `data.frame`;
the pandas index becomes a `date` column, because R has no separate notion
of an index.

In [ ]:
%%R -i macro
cat('R received', nrow(macro), 'rows\n')
str(macro)

In [ ]:
%%R
fit <- lm(lrcons ~ lrgdp + realint, data = macro)
summary(fit)

R's diagnostic plots render straight into the Colab output cell:

In [ ]:
%%R
par(mfrow = c(2, 2))
plot(fit)

### Bring results back to Python

`-o` returns an R object to the Python side.

In [ ]:
%%R -o coefs
coefs <- as.data.frame(summary(fit)$coefficients)

In [ ]:
coefs

## 5. Compare Python and R on the same model

`compare_ols` runs the specification in every engine available. On Colab
that is Python and R; on a Windows machine with all five installed, the
same line returns five columns.

In [ ]:
cmp = econenv.compare_ols(macro, 'lrcons ~ lrgdp + realint')
cmp

In [ ]:
cmp.coefficients()

The coefficients agree to machine precision. Any information criteria
that differ are reported with the reason — statsmodels uses $-2\ell + 2k$
while R counts $\sigma^2$ as a parameter — rather than being quietly
reconciled.

## 6. Record what produced this

Colab runtimes are disposable, which makes a snapshot more useful here
than anywhere else: it pins the versions that generated these numbers.

In [ ]:
econenv.snapshot()

---

## Next

- [The full five-engine notebook](https://github.com/merwanroudane/econenv/blob/main/examples/10_real_data_all_engines.ipynb) — Python, R, Stata, EViews **and** MATLAB, for a local Windows machine
- [Documentation site](https://merwanroudane.github.io/econenv/)
- [Installation & User Guide (PDF)](https://github.com/merwanroudane/econenv/blob/main/docs/guide/econenv-guide.pdf)
- [EViews commands for GUI users](https://github.com/merwanroudane/econenv/blob/main/docs/engines/eviews-commands.md)

*EconEnv — Dr Merwan Roudane — <https://github.com/merwanroudane/econenv>*